# BRCA all omics benchmark
- benchmark for the brca dataset
- take
    - [x] mRNA
    - [x] CNA
    - [x] miRNA
    - [ ] DNA methylation data
- benchmark several prediction methods on them, using
    - [ ] early integration
    - [ ] late integration

In [45]:
import numpy as np

# Assuming X is your input matrix with shape (n_omics, n_samples, n_features)
# Example matrix for demonstration
X = np.array([
    [[1, 2, 3], [4, 5, 6], [7, 8, 9]], # Omics 1
    [[10, 11, 12], [13, 14, 15], [16, 17, 18]], # Omics 2
    [[20, 21, 22], [23, 24, 25], [26, 27, 28]], # Omic 3
    [[20, 21, 22], [23, 24, 25], [26, 27, 28]] # Omic 4
])

X = [
    np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]]),
    np.array([[10, 11, 12], [13, 14, 15], [16, 17, 18]]),
    np.array([[20, 21], [23, 24], [26, 27]])
    
]

def concat_features(X):
    """
    Args:
        X (np.array): array with omic channels of shape (n_omics, n_samples, n_features)
    Returns:
        X (np.array): with shape (n_samples, n_features*n_omics)
    """

    return np.hstack(X)

X_reshaped = concat_features(X)

# print(X)
# # print("Original shape:", X.shape)
# # print("Transposed shape:", X_transposed.shape)
# print("Reshaped shape:", X_reshaped.shape)

X, X_reshaped

([array([[1, 2, 3],
         [4, 5, 6],
         [7, 8, 9]]),
  array([[10, 11, 12],
         [13, 14, 15],
         [16, 17, 18]]),
  array([[20, 21],
         [23, 24],
         [26, 27]])],
 array([[ 1,  2,  3, 10, 11, 12, 20, 21],
        [ 4,  5,  6, 13, 14, 15, 23, 24],
        [ 7,  8,  9, 16, 17, 18, 26, 27]]))

In [2]:
%load_ext autoreload
%autoreload 2

In [69]:
import polars as pl
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

from bipartite_gnn.graph_visualizatons import visualize_graph, visualize_embeddings
from baseline_evals.feature_selection import variational_selection
from baseline_evals.knn_eval import knn_eval
from baseline_evals.svm_eval import svm_lin_eval, svm_rbf_eval
from baseline_evals.xgboost_eval import xgboost_eval
from baseline_evals.mlp_eval import mlp_eval

In [22]:
null_vals = ["NA"]
mrna = pl.read_csv("BRCA_PROCESSED_DATA/mrna.tsv", separator="\t", null_values=null_vals)
cna = pl.read_csv("BRCA_PROCESSED_DATA/cnvth.tsv", separator="\t", null_values=null_vals)
mirna = pl.read_csv("BRCA_PROCESSED_DATA/mirna.tsv", separator="\t", null_values=null_vals)

In [32]:
labels = pl.read_csv("BRCA_PROCESSED_DATA/labels.tsv", separator="\t")
le = LabelEncoder()
le.fit(labels["PAM50_mRNA_nature2012"].to_list())
y = le.transform(labels["PAM50_mRNA_nature2012"].to_list())
labels

sampleID,PAM50_mRNA_nature2012
str,str
"""TCGA-A1-A0SD-0…","""Luminal A"""
"""TCGA-A1-A0SE-0…","""Luminal A"""
"""TCGA-A1-A0SH-0…","""Luminal A"""
"""TCGA-A1-A0SJ-0…","""Luminal A"""
"""TCGA-A1-A0SK-0…","""Basal-like"""
…,…
"""TCGA-E2-A1B4-0…","""Luminal A"""
"""TCGA-E2-A1B5-0…","""Basal-like"""
"""TCGA-E2-A1B6-0…","""Luminal A"""


In [29]:
# ensure that the omic channels are alined with the labels and with each other
assert mrna.columns[1:] == cna.columns[1:] == mirna.columns[1:] == labels["sampleID"].to_list()

In [26]:
# TODO, join genes with identical features into a single vertex
def find_identical_rows(matrix):
    # Convert the matrix to a structured array
    structured_array = np.core.records.fromarrays(matrix.transpose())
    
    # Find unique rows and their corresponding labels
    _, labels = np.unique(structured_array, return_inverse=True)
    
    # Group indices by labels
    unique_labels, indices = np.unique(labels, return_index=True)
    identical_rows_indices = [np.where(labels == label)[0] for label in unique_labels]
    
    return identical_rows_indices

# Example usage
matrix = np.array([[1, 2], [3, 4], [1, 2], [5, 6], [3, 4]])
identical_rows_indices = find_identical_rows(matrix)

for indices in identical_rows_indices:
    print(f"Identical rows at indices: {indices}")

# identical_rows = find_identical_rows(cna_m)

Identical rows at indices: [0 2]
Identical rows at indices: [1 4]
Identical rows at indices: [3]


In [56]:
# apply individual feature selection

X_mrna = mrna[:,1:].to_numpy().T
X_cna = cna[:,1:].to_numpy().T
X_mirna = mirna[:,1:].to_numpy().T

# mrna_mask, mrna_idx = variational_selection(X_mrna, y, 200)
# cna_mask, cna_idx = variational_selection(X_cna, y, 200)
# mirna_mask, mirna_idx = variational_selection(X_mirna, y, 100)

# mrna_mask.shape
# X_mrna.shape, X_cna.shape
X = np.hstack([X_mrna, X_cna, X_mirna])
X.shape
# X_mrna = X_mrna[:, mrna_mask]
# X_cna = X_cna[:, cna_mask]
# X_mirna = X_mirna[:, mirna_mask]

# # concat the features
# X = np.hstack([X_mrna, X_cna, X_mirna])
# X.shape

(483, 43982)

In [62]:
# run benchmarks
knn_eval(X, y, n_features=1000)

| KNN | 0.66 +/- 0.02 | 0.56 +/- 0.04 | 0.62 +/- 0.03 |
study.best_value=0.6167785914904695, study.best_params={'n_neighbors': 1}


In [63]:
svm_lin_eval(X, y, n_features=1000)

Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
| LIN SVM | 0.82 +/- 0.02 | 0.81 +/- 0.02 | 0.82 +/- 0.02 |
study.best_value=0.82394274728502, study.best_params={'C': 0.0012062927436782822, 'class_weight': None}


{'acc': 0.8234482758620689,
 'f1_macro': 0.8146722196381695,
 'f1_weighted': 0.82394274728502,
 'acc_std': 0.024128076806256414,
 'f1_macro_std': 0.024158775154244774,
 'f1_weighted_std': 0.023747662716833852}

In [64]:
svm_rbf_eval(X, y, n_features=1000)

Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
| RBF SVM | 0.81 +/- 0.02 | 0.79 +/- 0.02 | 0.80 +/- 0.02 |
study.best_value=0.8024684394738328, study.best_params={'C': 8.681172078950329, 'gamma': 0.0010065527711077555, 'class_weight': 'balanced'}


{'acc': 0.8060344827586208,
 'f1_macro': 0.7858921149649299,
 'f1_weighted': 0.8024684394738328,
 'acc_std': 0.02023568033500948,
 'f1_macro_std': 0.021351915435578694,
 'f1_weighted_std': 0.021399159772430745}

In [65]:
xgboost_eval(X, y, n_features=1000)

0 / 100
1 / 100
Pruning trial
2 / 100
3 / 100
Pruning trial
4 / 100
Pruning trial
5 / 100
Pruning trial
6 / 100
Pruning trial
7 / 100
Pruning trial
8 / 100
Pruning trial
9 / 100
Pruning trial
10 / 100
11 / 100
12 / 100
13 / 100
14 / 100
15 / 100
16 / 100
17 / 100
18 / 100
Pruning trial
19 / 100
Pruning trial
20 / 100
Pruning trial
21 / 100
22 / 100
Pruning trial
23 / 100
24 / 100
Pruning trial
25 / 100
Pruning trial
26 / 100
Pruning trial
27 / 100
Pruning trial
28 / 100
Pruning trial
29 / 100
30 / 100
31 / 100
32 / 100
33 / 100
34 / 100
35 / 100
36 / 100
37 / 100
38 / 100
Pruning trial
39 / 100
Pruning trial
40 / 100
Pruning trial
41 / 100
Pruning trial
42 / 100
43 / 100
44 / 100
45 / 100
Pruning trial
46 / 100
Pruning trial
47 / 100
48 / 100
49 / 100
50 / 100
Pruning trial
51 / 100
52 / 100
53 / 100
Pruning trial
54 / 100
55 / 100
56 / 100
Pruning trial
57 / 100
58 / 100
Pruning trial
59 / 100
60 / 100
Pruning trial
61 / 100
62 / 100
63 / 100
64 / 100
65 / 100
66 / 100
Pruning trial
6

In [ ]:
# mlp
train_temp_idx, test_idx = train_test_split(
    np.arange(X.shape[0]),
    test_size=0.2,
    stratify=y,
    random_state=3
)
train_idx, val_idx = train_test_split(
    train_temp_idx, 
    test_size=0.2,
    stratify=y[train_temp_idx],                   
    random_state=3
)

X_train = X[train_idx]
y_train = y[train_idx]
X_train = X[train_idx]
y_train = y[train_idx]
X_train = X[train_idx]
y_train = y[train_idx]
# split data


In [71]:
mlp_eval(X, y, n_trials=1)

0 / 1


AttributeError: 'MLPTrainer' object has no attribute 'fit'